<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [7]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')

<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [ ]:
# File path for the document

file_path = r"lab-intro-rag\ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [34]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [20]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [21]:
load_dotenv()

True

In [22]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [24]:
from langchain_community.vectorstores import Chroma

In [25]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [ ]:
user_question = "Does AI produce new inequalities?"
#"What kind of policies does the book suggest to democratically regulate AI?"
#"Is AI accessible for everyone?"
#"How are political decisions influenced by AI?"
#"How does Artificial Intelligence shapes society?"
#"How are power structures shaped by digital and social media?"
#"What is the most unethical influence, the internet, digital, and social media has on our society?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [47]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:

Document 2:
nce because they are not the expression of a hypothesis but sim-
ply an extrapolation from a set of data points. And yet AI is performative; by 
intervening in the world and reorganising the phenomena it claims to describe, 
it brings our experience into closer alignment with its constructs (Mackenzie 
2008). This invalidates the methods as a form of statistical insight but makes it 
very effective as a mode of subjection.
Automated Segregation
As the logic of AI migrates from the abstract mathematical space of tensors 
to the space of real social tensions, it comes to bear in specific ways. First and 
foremost of these is automated segregation.
There is nothing personal about the predictions of AI – at root, they are always 
some form of labelling in terms of ‘people/objects like you’ . As an offshoot of 
the statistics family tree, machine learning’s classifications are governed by the
Document 3:
race; how it seems to always be the poorest 
and most marginal

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [28]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [29]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


In [35]:
formatted_context   

'\n\nContent:\nWatzlawick, P . 1964. An Anthology of Human Communication. Palo Alto, CA: \nScience and Behavior Books.\nWeizenbaum, J. 1976. Computer Power and Human Reason: From Judgment to \nCalculation. San Francisco: W . H. Freeman.\n\n\nContent:\nis negated – done so by conflations – and when the difference between human \nand machine is affirmed at the cost of their unity that is negated – done so by \ndisconnections. The way out is the establishment of a relation through affirm -\ning both the identity of, and the difference between, the two sides – as done by\n\n\nContent:\n-right/.\nMackenzie, D. 2008. An Engine, Not a Camera: How Financial Models Shape \nMarkets. (1st edn.). Cambridge, MA: The MIT Press.\nMalik, M. M. 2020. A Hierarchy of Limitations in Machine Learning. \nArXiv:2002.05193 [Cs, Econ, Math, Stat] , February. http://arxiv.org \n/abs/2002.05193.\nMarcus, G. 2018. Deep Learning: A Critical Appraisal. ArXiv:1801.00631 [Cs, \nStat], January. http://arxiv.org/abs/18

<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [48]:
prompt = f"""
## SYSTEM ROLE
You are a chatbot specializing in **critical perspectives on Artificial Intelligence** and its affects on society, labour, democracy, free will, who benefits from it, and whos interests are served. 
Your central theme is **disbalanced power through AI** and you draw from the critical, sociological, and politiocal analyses found in *AI for Everyone?*.

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from *AI for Everyone?*:
'''
{formatted_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "I am not able to provide scientific content based on the context."
   - Emphasise power relations, political economy, and social justice.
   - Distinguish between what AI is (technically) and what AI does in society (politically/economically).
   - When describing AI narratives, identify and highlight potential myths, assumptions or ideologies. 

2. **Transparency**:
   - Reference the author's name and page numbers when providing information.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use simple explanations for complex theories.
   - Provide examples from the book. 
   - Avoid technological deternism.
   - Format your response in Markdown for readability.

## TASK
1. Answer the user's question **directly** if possible.
2. Point the user to relevant parts of the documentation.
3. Provide the response in the following format:

## RESPONSE FORMAT
'''
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Source**:
• [Book Title], [Author], Page(s): [...]
'''
"""
print("Prompt constructed.")

Prompt constructed.


In [49]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [50]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.6,  # Increase creativity
    'max_tokens': 4000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.5,  # Reduce repetition
    'presence_penalty': 0.6   # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [51]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [52]:
answer = completion.choices[0].message.content
print(answer)

'''
# Does AI Produce New Inequalities?

AI does indeed produce new inequalities, especially when examined through the lens of power relations, political economy, and social justice. The book *AI for Everyone?* highlights several areas where AI contributes to inequality:

1. **Global Inequality**: There is a noted divide between the Global North and Global South in terms of access to AI technologies and benefits (Page 239, 248-250). This disparity can exacerbate existing economic and technological gaps.

2. **Labour Inequality**: The rise of AI has transformed labor markets, often leading to precarious work conditions such as those experienced by Brazilian workers on global AI platforms (Page 247-266). This shift can result in job displacement and increased reliance on gig work.

3. **Bias in AI Systems**: Bias is a significant issue in AI systems, which can lead to discrimination and further entrench societal biases (Page 114, 214). These biases are often embedded within algorithms th

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [53]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [54]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [58]:
query_keywords = ["inequality", "unethical", "power" ] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
PART 3
AI Power and Inequalities
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [ ]:
# File path for the document

file_path = r"lab-intro-rag\bell_hooks_All_about_Love_New_Visions_William_Morrow_Paperbacks.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>



In [64]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

244

<h3 style="color: #FF8C00;">Pages into Chunks</h3>





In [71]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

451

<h3 style="color: #FF8C00;">Embeddings</h3>

In [77]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [92]:
user_question = "What defines love?"
#"How can we find back to mutual love?"
#"How are traumatic events in childhood shaping our experience of love?"
# "What is the patriarchal impact on love?"
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [79]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
. If we were constantly remembering 
that love is as love does, we would not use the word in a 
manner that devalues and degrades its meaning. When we 
are loving we openly and honestly express care, affection, 
responsibility, respect, commitment, and trust. 
Definitions are vital starting points for the imagination. 
What we cannot imagine cannot come into being. A good 
definition marks our starting point and lets us know where 
we want to end up. As we move toward our desired des­
tination we chart the journey, creating a map . We need a 
map to guide us on our journey to love-starting with the 
place where we know what we mean when we speak of 
love. 
I 4
Document 2:
ow much easier it would be for us to learn 
how to love if we began w ith a shared definition. Th e 
word "love" is most often defined as a noun, yet all the 
more astute theorists of love acknowledge that we would 
all love better if we used it as a verb. I spent years search­
ing for a meaning ful defini

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [80]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [81]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


In [82]:
formatted_context   

'\n\nContent:\nALL ABOUT LOVE \nhe loved her so much. If we were constantly remembering \nthat love is as love does, we would not use the word in a \nmanner that devalues and degrades its meaning. When we \nare loving we openly and honestly express care, affection, \nresponsibility, respect, commitment, and trust. \nDefinitions are vital starting points for the imagination. \nWhat we cannot imagine cannot come into being. A good \ndefinition marks our starting point and lets us know where \nwe want to end up. As we move toward our desired des\xad\ntination we chart the journey, creating a map . We need a \nmap to guide us on our journey to love-starting with the \nplace where we know what we mean when we speak of \nlove. \nI 4\n\n\nContent:\nthey use the word "love." \nImagine how much easier it would be for us to learn \nhow to love if we began w ith a shared definition. Th e \nword "love" is most often defined as a noun, yet all the \nmore astute theorists of love acknowledge that we

<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [83]:
prompt = f"""
## SYSTEM ROLE
You are a chatbot that helps users to apply the concepts from *bell hooks: All About Love - New Visions* to real-life situations. 
Your purpose is to translate the books' philosophical ideas into clear explanations and gentle, actionable reflections. 

## USER QUESTION
The user has asked:
"{user_question}"

## CONTEXT
Here is the relevant content from *bell hooks: All About Love - New Visions*:
'''
{formatted_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "Based on unsufficient context, I don't feel comfortable answering your question."
   - Root all ethical interpretations in hooks' explicit arguements.
   - Help the user to understand the origin of today's distorted practice of love by focusing on trauma (generational, partriarchy, etc.).
   - Answer in a gentle tone and meet the user on an emotional level.

2. **Transparency**:
   - clarify when the bot is actually quoting vs. when offering context or own interpretation. 

3. **Clarity**:
   - Use simple and warm explanations for complex theories.
   - Provide examples from the book. 
   - Avoid technological deternism.
   - Format your response in Markdown for readability.

## TASK
1. Answer the user's question **directly** if possible.
2. Point the user to relevant parts of the documentation.
3. Provide the response in the following format:

## RESPONSE FORMAT
'''
# [Brief Title of the Answer]
[Answer in simple, clear text.]

**Source**:
• [Book Title], Page(s): [...]
'''
"""
print("Prompt constructed.")

Prompt constructed.


### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [86]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.8,  # Increase creativity
    'max_tokens': 4000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.5,  # Reduce repetition
    'presence_penalty': 0.6   # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [93]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [94]:
answer = completion.choices[0].message.content
print(answer)

'''
# Understanding Love Through bell hooks' Lens

In *bell hooks: All About Love - New Visions*, love is defined not as a mere feeling but as an active choice and commitment. Hooks emphasizes that "love is as love does," meaning it involves openly expressing care, affection, responsibility, respect, commitment, and trust. This perspective suggests that love is more about action than emotion.

To further explain this concept, hooks draws on the work of M. Scott Peck, who defines love as "the will to extend one's self for the purpose of nurturing one's own or another's spiritual growth." This means love requires both intention and action—choosing to engage in behaviors that nurture growth.

Hooks argues that defining love clearly helps us understand what it truly means and how we can practice it authentically. Instead of thinking of love only as a profound feeling or attraction (as often emphasized in romantic contexts), she encourages viewing it as a deliberate choice involving various